# SAADE–AUA v3.1 · повний експеримент без експертної оцінки

**Мета:** не просто прочитати всі WAV, а правильно зіставити кожен запис з AUA `metadata_audio.csv`, окремим реєстром DRSSU (`generated`), реєстром TTS (`model_name`, `voice_id`) та першоджерелом. Потім виконати 21 контрольований сценарій, E1/E2/E3, порівняння моделей і відтворення аудіо.

**Обмеження:** відсутність matched real/fake в межах зіставних джерел **не можна виправити лише кодом**. `pilot` виконує технічний експеримент із явним попередженням; `research` не дозволяє незалежний hidden test за відсутності такого контролю. `S1108/S1109` за `source_group_unknown` — **не підтверджені мовці**. Експертна валідація в цю версію навмисно не включена.

Старі v1/v2 не змінюються. Модулі **виключно v3** (`saade_v3_*.py`) повинні лежати поруч із ноутбуком. Вихідні аудіо AUA не змінюються.


## 0. Параметри запуску — налаштуйте перед Run All

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/jovyan/work/audio_real_fake')
DATA_ROOT = PROJECT_ROOT / 'AUA'
RESULTS_ROOT = PROJECT_ROOT / 'results' / 'SAADE_AUA_2026_v3_1_metadata_fixed'

EXPERIMENT_MODE = 'pilot'       # 'pilot' або 'research'
RUN_TRAINING = True            # False: тільки каталог, split, ілюстрації та контроль
RUN_REPEATS = False            # True: три повні незалежні навчання; рекомендовано для статті
RUN_FINAL_HIDDEN_TEST = False # УВАГА: тільки після фіксації протоколу та усунення конфаундингу

SEEDS = (42, 123, 777)
EPOCHS_PILOT = 4
EPOCHS_RESEARCH = 18
MAX_PARENT_FILES = 900
TRAIN_ONLY_UNKNOWN_CAP = 150  # DRSSU/FLEURS з невідомими voice/speaker — лише TRAIN
TARGET_SEGMENTS = 1200
SECONDS_PER_SEGMENT = 4.0
MODELS = ('spectrogram_cnn', 'raw_wave_cnn', 'mfcc_svm')
# Для окремого повного прогону після встановлення оригінальних реалізацій:
# MODELS = ('spectrogram_cnn', 'raw_wave_cnn', 'mfcc_svm', 'aasist', 'wavlm')

assert EXPERIMENT_MODE in ('pilot','research')
if EXPERIMENT_MODE=='research':
    assert len(SEEDS) in (3,4,5)
assert DATA_ROOT.resolve() == (PROJECT_ROOT/'AUA').resolve()
assert RESULTS_ROOT.resolve().is_relative_to((PROJECT_ROOT/'results').resolve()), 'Result folder must be under audio_real_fake/results'
assert not any('melnyk' in part.lower() for part in RESULTS_ROOT.resolve().parts)
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
print('AUA:',DATA_ROOT)
print('OUT:',RESULTS_ROOT)
print('Режим:',EXPERIMENT_MODE)


### 0.1. Підключення пакетів та **правильних** модулів v3

In [ ]:
import sys, importlib, shutil, importlib.util
from dataclasses import replace
import numpy as np, pandas as pd
from IPython.display import display, Markdown
HERE=Path.cwd().resolve()
sys.path.insert(0,str(HERE))
required_files=['saade_v3_core.py','saade_v3_data.py','saade_v3_methods.py','saade_v3_visuals.py']
missing_files=[x for x in required_files if not (HERE/x).is_file()]
if missing_files:raise FileNotFoundError('Файли v3 мають лежати поруч із ноутбуком: '+str(missing_files))
importlib.invalidate_caches()
import saade_v3_core as S
import saade_v3_data as D
import saade_v3_methods as M
import saade_v3_visuals as V
for name,m in [('CORE',S),('DATA',D),('METHODS',M),('VISUALS',V)]:print(name,m.__file__)
required=['numpy','pandas','scipy','soundfile','sklearn','matplotlib','statsmodels','torch','joblib']
miss=[x for x in required if importlib.util.find_spec(x) is None]
if miss:raise ImportError('Встановіть через %pip install: '+', '.join(miss))
if shutil.which('ffmpeg') is None:raise RuntimeError('Потрібен ffmpeg для справжнього MP3-кодування')
assert DATA_ROOT.is_dir(),'Невірний DATA_ROOT'
print('CUDA доступна:',__import__('torch').cuda.is_available())


### 0.2. Конфігурація сценаріїв, моделі та відтворюваність

In [ ]:
cfg=S.Config(
    root=str(DATA_ROOT),out=str(RESULTS_ROOT),seed=42,scenario_seed=1001,
    scenario_profile='v2_5level', seconds=SECONDS_PER_SEGMENT,sample_rate=16000,
    min_seconds=2.0,max_items=MAX_PARENT_FILES,epochs=(EPOCHS_RESEARCH if EXPERIMENT_MODE=='research' else EPOCHS_PILOT),
    batch_size=12, models=MODELS, bootstraps=300, allow_weak_groups=False,
    augmentation_probability=0.5, test_finalization=False)
assert len(S.available_levels(cfg.scenario_profile))==5
print('Сценаріїв:',1+len(S.CATEGORIES)*len(S.available_levels(cfg.scenario_profile)))
print('Архітектури:',cfg.models,'| епохи:',cfg.epochs)
S.save_json(RESULTS_ROOT/'00_RUN_CONFIG.json',dict(mode=EXPERIMENT_MODE,run_training=RUN_TRAINING,
  run_repeats=RUN_REPEATS,run_hidden=RUN_FINAL_HIDDEN_TEST,models=MODELS,seeds=SEEDS,
  epochs=cfg.epochs,scenario_profile=cfg.scenario_profile,metadata_policy='v3.1 primary metadata + DRSSU selection_rank and SHA'))


## 1. Перебудова каталогу AUA за первинними метаданими

Мітка `fake=1` для DRSSU перевіряється за `00_sources_and_licenses/audio/drssu/metadata.csv → generated=True`. Для нової TTS — за реєстром `model_name + voice_id`. Common Voice, FLEURS та власні записи мають `real=0` **за походженням**, а не через filename. Усі джерела з невстановленими зв’язками отримують `CHECK` / не допускаються до експерименту. `approval_status=confirmed` не прирівнюється до експертної перевірки звуку.


In [ ]:
catalog=D.build_verified_catalog(cfg)
dr=catalog[catalog.source.eq('DRSSU')]
if len(dr):
    print('DRSSU files:',len(dr),'| source-linked:',int(dr.primary_source_meta_found.sum()),
          '| fake labelled:',int(dr.label.eq(1).sum()),'| usable:',int(dr.usable.sum()))
    if not dr.primary_source_meta_found.any():
        raise RuntimeError('DRSSU metadata not linked: check selection_rank and source SHA-256')
source_table=pd.read_csv(RESULTS_ROOT/'01_sources_summary.csv')
display(source_table)
print('Підтверджених для обробки файлів:',int(catalog.usable.sum()))
print('Невідомих осіб/голосів:',int(catalog.identity_status.eq('unknown_identity').sum()))
print('Недостовірних/неприв’язаних до метаданих:',int(catalog.label.isna().sum()))
display(pd.crosstab(catalog.source,catalog.label,dropna=False))
display(catalog[catalog.warning.ne('')][['relative_path','source','speaker_id_basis','identity_status','warning']].head(20))


### 1.1. Перевірка джерело × клас; ризик конфаундингу

In [ ]:
wide,matched_text,quality=M.source_balance_audit(catalog,cfg,tag='authoritative_catalog')
display(wide)
print('Є джерела з обома класами:',quality['matched_sources'])
if not quality['matched_sources']:
    display(Markdown('**ПОПЕРЕДЖЕННЯ: жодне джерело не містить водночас REAL і FAKE. Поки це лише пілотне міжджерельне порівняння, а не доведена детекція синтетичного мовлення.**'))


## 2. Первинні розбиття — незалежність мовців і синтетичних голосів

Split виконується **до** розбивки довгих записів на 4-секундні сегменти. Якщо для DRSSU невідомі `generator/voice_id`, а для FLEURS `speaker_id_basis=source_group_unknown`, ці джерела не потрапляють до validation/hidden. Це **не** означає, що вони не використовуються: у pilot вони можуть додаватися до train. Первинні/синтетичні групи з відомими ID не перетинаються між вибірками.


In [ ]:
policy='strict_source' if EXPERIMENT_MODE=='research' else 'exploratory'
try:
    parents,parents_summary=D.prepare_parent_splits(catalog,cfg,
        max_parent_files=MAX_PARENT_FILES,unknown_train_cap=TRAIN_ONLY_UNKNOWN_CAP,policy=policy)
except ValueError as exc:
    if EXPERIMENT_MODE=='research':
        print('СТРОГИЙ РЕЖИМ ЗАБЛОКОВАНО:',exc)
        raise
    raise
display(parents_summary)
readiness=D.data_readiness_report(catalog,parents,cfg,policy)
display(readiness)
assert not ((parents.split.ne('train'))&parents.split_role.eq('train_only')).any()


## 3. Неперекривні аудіосегменти та фіксація split

Кількість сегментів — це **не** кількість незалежних мовців. Якщо вихідних записів недостатньо, програма не вигадує додаткові групи. Hidden test фізично залишається в маніфесті, але не оцінюється без окремого рішення.


In [ ]:
splits,segment_counts,segment_info=M.expand_nonoverlapping_segments(
    parents,cfg,target=TARGET_SEGMENTS,max_per_parent=30)
display(segment_counts)
print(segment_info)
for sn in ('train','validation','hidden_test'):
    d=splits[splits.split.eq(sn)]
    assert d.label.nunique()==2,f'У {sn} тільки один клас — навчання/оцінювання некоректне'
# Independently verify inherited group and source SHA integrity
for col in ('group_id','parent_file_id','source_sha256','working_sha256'):
    v=splits[splits[col].fillna('').astype(str).str.strip().ne('')]
    assert (v.groupby(col).split.nunique()<=1).all(),f'Leakage by {col}'
lock=S.lock_hidden_test(splits,cfg)
print('SPLIT LOCK:',lock)


## 4. Сценарії: таблиця всіх параметрів, графіки, **аудіоплеєри**

Повний набір: `baseline + 4 категорії × 5 рівнів = 21`. Впливи: Gaussian noise, MP3 стиснення, синтетична реверберація і low-pass. Вони зберігають еталонну мітку класу. Для одного REAL та FAKE з validation показуємо **всі 21 звукові варіанти**, осцилограми, спектрограми та параметри.


In [ ]:
guide=V.draw_scenario_guide(cfg,tag='validation')
examples=V.pick_examples(splits,'validation')
for record in examples:
    print('REAL' if int(record.label)==0 else 'FAKE',record.relative_path)
    V.show_audio_gallery(record,cfg,show_players=True,show_figures=True,tag='validation')


### 4.1. Інтерактивний аудіоплеєр (необов’язкова бібліотека ipywidgets)

In [ ]:
player=V.make_interactive_player(splits,cfg,'validation')


## 5. E1 — Static, навчання базових моделей

Усі моделі навчаються **на новому split** і проходять однакову validation зі сценаріями. `Spectrogram CNN`, `Raw Wave CNN`, `MFCC-SVM` — три базові архітектури. Для сильних моделей у параметрах можна вибрати `aasist`, `rawnet2`, `wavlm`, але вони використовують **справжні** зовнішні реалізації й потребують відповідних залежностей/ваг.


In [ ]:
models={};infos={};base_parts=[]
if RUN_TRAINING:
    for name in cfg.models:
        print('Навчання E1 Static:',name)
        m,meta=S.train_one(splits,cfg,name,'static')
        models[(name,'static')]=m;infos[(name,'static')]=meta
        preds=S.run_scenario_predictions(splits,m,name,'static',cfg,
                    threshold=meta['threshold'],split='validation')
        base_parts.append(preds)
    base_val=pd.concat(base_parts,ignore_index=True)
    base_val.to_csv(RESULTS_ROOT/'04_validation_E1_static.csv',index=False)
    print('E1 predictions:',len(base_val))
else:
    print('RUN_TRAINING=False: лише каталог і прослуховування; E1/E2/E3 пропущено')


## 6. E2 — Random; E3 — Adaptive

Для E3 ваги вибору категорій обчислюються **лише з помилок E1 на validation**. Для E2 розподіл категорій рівноймовірний. Усі моделі навчаються заново для кожного режиму за однакової кількості епох. Дані hidden test не використовуються.


In [ ]:
if RUN_TRAINING:
    weights_by={name:S.validation_difficulty(base_val[base_val.model.eq(name)]) for name in cfg.models}
    weights={c:float(np.mean([weights_by[name][c] for name in cfg.models])) for c in S.CATEGORIES}
    total=sum(weights.values());weights={c:v/total for c,v in weights.items()}
    S.save_json(RESULTS_ROOT/'05_adaptive_weights.json',weights)
    print('Adaptive weights:',weights)
    prediction_parts=[base_val]
    for name in cfg.models:
        for regime in ('random','adaptive'):
            print('Навчання',name,regime)
            m,meta=S.train_one(splits,cfg,name,regime,
                         adaptive_weights=weights if regime=='adaptive' else None)
            models[(name,regime)]=m;infos[(name,regime)]=meta
            p=S.run_scenario_predictions(splits,m,name,regime,cfg,
                         threshold=meta['threshold'],split='validation')
            prediction_parts.append(p)
    validation_predictions=pd.concat(prediction_parts,ignore_index=True)
    validation_predictions.to_csv(RESULTS_ROOT/'04_val_predictions_ALL.csv',index=False)
    print('Усі прогнозування:',len(validation_predictions))


## 7. Повне тестування **КОЖНОГО СЦЕНАРІЮ**

Для всіх моделей і E1/E2/E3 відображаються: Balanced Accuracy, EER, ROC-AUC, F1, хибні спрацювання, пропуски FAKE, ASRI, залежність від рівня, деградація проти оригіналу, confusion matrix. Окремо показуються **рішення моделей щодо одного й того самого аудіо** у всіх 21 сценарії.


In [ ]:
if RUN_TRAINING:
    scenario_metrics,asri_table,coverage=V.report_scenario_tests(
        validation_predictions,cfg,tag='validation',show=True)
    display(asri_table.sort_values(['model','regime']))
    for record in examples:
        V.show_one_record_decisions(validation_predictions,record,cfg,tag='validation')
    V.plot_roc_for_selected(validation_predictions,cfg,model=cfg.models[0],
                            regime='adaptive',tag='validation')


## 8. Статистика: cluster bootstrap і GEE з взаємодіями

Статистика враховує групи мовців/голосів; **не трактує 21 сценарій одного запису як 21 незалежну людину**. Неповнота незалежних синтетичних груп залишається обмеженням. Результати validation після адаптації — дослідницькі, не остаточне незалежне підтвердження.


In [ ]:
if RUN_TRAINING:
    stats=M.interaction_statistics(validation_predictions,cfg,'validation')
    print(stats)
    boot=[]
    for name in cfg.models:
        res=S.paired_group_bootstrap_delta(validation_predictions,name,
                regime_a='random',regime_b='adaptive',B=cfg.bootstraps,seed=cfg.seed)
        res['model']=name;boot.append(res)
    display(pd.DataFrame(boot))
    pd.DataFrame(boot).to_csv(RESULTS_ROOT/'11_bootstrap_v3.csv',index=False)


## 9. Повні повтори 3–5 seeds для статті

Для публікації увімкніть `RUN_REPEATS=True`, бажано з `EXPERIMENT_MODE='research'` **після виправлення джерело×клас**. На кожному seed E1/E2/E3 навчаються заново. Перед зміною режиму створіть **нову вихідну папку**, щоб не змішати протоколи. Змінні data/source не залежать від seed; контрольні сценарії фіксуються окремим `scenario_seed`.


In [ ]:
if RUN_TRAINING and RUN_REPEATS:
    replicates,replicate_summary,all_seed_predictions=M.seed_replicates(
        cfg,splits,seeds=SEEDS,models=cfg.models)
    display(replicate_summary)
else:print('Режим повторів не ввімкнено. Для статті потрібні 3–5 seeds.')


## 10. Фіксація протоколу та **захищений** hidden test

До незалежного тестування мають бути виконані три умови: (1) зіставні real/fake в тому самому джерелі з доказовими метаданими, (2) 3–5 завершених повторів, (3) зафіксовані пороги, моделі, split, параметри сценаріїв. **Якщо обмеження не усунуті, запуск hidden блокується, а не видається псевдопідтвердження**. Попередні hidden v1/v2 не використовуються для вибору параметрів.


In [ ]:
if RUN_TRAINING:
    locked=S.save_locked_protocol(cfg,splits,infos)
    print('Protocol locked:',locked.get('dataset_hash'))
if RUN_FINAL_HIDDEN_TEST:
    assert RUN_TRAINING and RUN_REPEATS, 'Для final hidden необхідні навчання й 3–5 seeds'
    if not quality['matched_sources']:
        raise RuntimeError('HIDDEN TEST BLOCKED: Немає source-matched REAL/FAKE у поточному AUA')
    from dataclasses import replace
    # Mark one-shot before predictions; keep original v3 configuration unchanged
    M.reserve_hidden_evaluation(cfg,splits,expected_seed_protocols=SEEDS)
    hidden_preds,hidden_asri,hidden_summary=M.hidden_all_repeats(cfg,splits,seeds=SEEDS,models=cfg.models)
    print('Hidden predictions:',len(hidden_preds))
    display(hidden_summary)
    # Do NOT pool predictions across seeds as if they were independent recordings.
    for one_seed,one_pred in hidden_preds.groupby('training_seed'):
        V.report_scenario_tests(one_pred,cfg,tag=f'hidden_seed_{one_seed}',show=True)
else:print('Hidden test НЕ відкрито (запланована безпечна поведінка)')


## 11. Експорт і короткий висновок

Створюється ZIP із CSV, JSON, PNG та статистичними звітами. **Самі голосові записи й ваги моделей за замовчуванням не включаються** — для повторення експерименту вони залишаються на сервері. Інструкція щодо відтворення міститься у README.


In [ ]:
summary={'data_root':str(DATA_ROOT),'mode':EXPERIMENT_MODE,
    'catalog_files':len(catalog),'usable':int(catalog.usable.sum()),
    'unknown_identity_files':int(catalog.identity_status.eq('unknown_identity').sum()),
    'matched_source_count':len(quality['matched_sources']),
    'parent_files':len(parents),'segments':len(splits),'group_count':int(splits.group_id.nunique()),
    'scenario_count':21,'models':list(cfg.models),
    'trained':RUN_TRAINING,'repeats_run':RUN_TRAINING and RUN_REPEATS,
    'hidden_run':RUN_FINAL_HIDDEN_TEST,
    'expert_validation':'not_included',
    'limitation':'Source/class confounding remains if matched_source_count=0'}
S.save_json(RESULTS_ROOT/'FINAL_V3_SUMMARY.json',summary)
print(pd.Series(summary))
archive=S.zip_results(cfg,include_models=False)
print('Архів:',archive)


---
### Наукові обмеження, які слід вказати у статті

1. **Source/confounding.** Якщо `matched_source_count = 0`, показники відображають розрізнення джерел і можуть завищувати реальну детекцію TTS. Тоді дослідження — пілотне.
2. **Unverified voices.** Технічні `S1108`, `S1109` не є голосовими ID. Дані FLEURS/DRSSU з невідомими мовцями допускаються тільки до train у пілоті.
3. **Validation reuse.** Адаптивні ваги сформовані на validation, отже результати validation не підтверджують незалежну перевагу E3.
4. **Сценарії.** П'ять рівнів є контрольованими цифровими змінами, не виміряним відтворенням реальних кімнат/месенджерів.
5. **Експертне оцінювання не виконувалося.** Жодна метрика не є експертною валідацією.
